# Apache Airflow TaskFlow API: Automatic XCom Handling

## Overview

TaskFlow API eliminates manual XCom push/pull operations. Data passes between tasks through Python function return values and arguments, with Airflow handling serialization automatically.

### Key Mechanisms
- Return values automatically pushed to XCom
- Function arguments automatically pulled from XCom
- Support for complex data types (dicts, lists, pandas DataFrames)
- Type preservation across task boundaries

## 1. Basic Data Passing

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime

@dag(
    dag_id='basic_data_passing',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def basic_passing_dag():
    """Demonstrates fundamental data passing."""
    
    @task
    def produce_value():
        """Return value is automatically stored in XCom."""
        # No xcom_push() needed
        return 42
    
    @task
    def consume_value(number):
        """Argument is automatically retrieved from XCom."""
        # No xcom_pull() needed
        result = number * 2
        print(f'Received: {number}, Result: {result}')
        return result
    
    # Data flows through function call syntax
    value = produce_value()
    consume_value(value)

dag_instance = basic_passing_dag()

## 2. Complex Data Types

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime
import json

@dag(
    dag_id='complex_types_passing',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def complex_types_dag():
    """Passing dictionaries, lists, and nested structures."""
    
    @task
    def create_record():
        """Return complex dictionary."""
        return {
            'id': 12345,
            'name': 'John Doe',
            'email': 'john@example.com',
            'metadata': {
                'created': '2026-09-13',
                'source': 'api'
            }
        }
    
    @task
    def enrich_record(record):
        """Modify and extend dictionary."""
        record['status'] = 'processed'
        record['enriched_at'] = '2026-09-13T10:00:00'
        return record
    
    @task
    def validate_record(record):
        """Validate data structure."""
        required_fields = ['id', 'name', 'email', 'status']
        missing = [f for f in required_fields if f not in record]
        
        if missing:
            raise ValueError(f'Missing fields: {missing}')
        
        print(f'Valid record for {record["name"]}')
        return True
    
    record = create_record()
    enriched = enrich_record(record)
    validate_record(enriched)

dag_instance = complex_types_dag()

## 3. List Processing Patterns

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime

@dag(
    dag_id='list_processing',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def list_processing_dag():
    """Working with lists and collections."""
    
    @task
    def fetch_ids():
        """Return list of identifiers."""
        return [101, 102, 103, 104, 105]
    
    @task
    def process_batch(ids):
        """Process entire list."""
        results = []
        for id in ids:
            results.append({'id': id, 'status': 'processed'})
        print(f'Processed {len(results)} items')
        return results
    
    @task
    def aggregate_results(results):
        """Summarize batch results."""
        total = len(results)
        successful = sum(1 for r in results if r['status'] == 'processed')
        
        summary = {
            'total': total,
            'successful': successful,
            'success_rate': successful / total if total > 0 else 0
        }
        
        print(f'Summary: {summary}')
        return summary
    
    ids = fetch_ids()
    results = process_batch(ids)
    aggregate_results(results)

dag_instance = list_processing_dag()

## 4. Multiple Inputs to Single Task

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime

@dag(
    dag_id='multiple_inputs',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def multiple_inputs_dag():
    """Combine data from multiple upstream tasks."""
    
    @task
    def get_user_data():
        return {
            'user_id': 1001,
            'name': 'Alice',
            'tier': 'premium'
        }
    
    @task
    def get_transaction_data():
        return {
            'transaction_id': 5001,
            'amount': 299.99,
            'currency': 'USD'
        }
    
    @task
    def get_preferences():
        return {
            'notifications': True,
            'language': 'en',
            'timezone': 'UTC'
        }
    
    @task
    def create_complete_profile(user, transaction, prefs):
        """Merge data from three sources."""
        profile = {
            **user,
            'last_transaction': transaction,
            'preferences': prefs,
            'profile_complete': True
        }
        
        print(f'Profile created for {profile["name"]}')
        return profile
    
    user = get_user_data()
    transaction = get_transaction_data()
    prefs = get_preferences()
    
    create_complete_profile(user, transaction, prefs)

dag_instance = multiple_inputs_dag()

## 5. Pandas DataFrame Passing

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime

@dag(
    dag_id='pandas_dataframe_passing',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def pandas_dag():
    """Pass DataFrames between tasks."""
    
    @task
    def load_dataframe():
        """Create and return DataFrame."""
        try:
            import pandas as pd
            
            df = pd.DataFrame({
                'date': ['2026-09-01', '2026-09-02', '2026-09-03'],
                'revenue': [1000, 1500, 1200],
                'customers': [50, 75, 60]
            })
            
            print(f'DataFrame shape: {df.shape}')
            return df
        except ImportError:
            # Fallback without pandas
            return [
                {'date': '2026-09-01', 'revenue': 1000, 'customers': 50},
                {'date': '2026-09-02', 'revenue': 1500, 'customers': 75},
                {'date': '2026-09-03', 'revenue': 1200, 'customers': 60}
            ]
    
    @task
    def transform_dataframe(df):
        """Transform DataFrame."""
        try:
            import pandas as pd
            
            if isinstance(df, pd.DataFrame):
                df['revenue_per_customer'] = df['revenue'] / df['customers']
                df['date'] = pd.to_datetime(df['date'])
                print(f'Transformed shape: {df.shape}')
                return df
        except ImportError:
            pass
        
        # Fallback for list of dicts
        for row in df:
            row['revenue_per_customer'] = row['revenue'] / row['customers']
        return df
    
    @task
    def summarize_data(df):
        """Generate summary statistics."""
        try:
            import pandas as pd
            
            if isinstance(df, pd.DataFrame):
                summary = {
                    'total_revenue': df['revenue'].sum(),
                    'avg_revenue': df['revenue'].mean(),
                    'total_customers': df['customers'].sum(),
                    'rows': len(df)
                }
                print(f'Summary: {summary}')
                return summary
        except ImportError:
            pass
        
        # Fallback
        total_rev = sum(r['revenue'] for r in df)
        total_cust = sum(r['customers'] for r in df)
        return {
            'total_revenue': total_rev,
            'avg_revenue': total_rev / len(df),
            'total_customers': total_cust,
            'rows': len(df)
        }
    
    df = load_dataframe()
    transformed = transform_dataframe(df)
    summarize_data(transformed)

dag_instance = pandas_dag()

## 6. Conditional Data Flow

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime

@dag(
    dag_id='conditional_flow',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def conditional_dag():
    """Route data based on conditions."""
    
    @task
    def check_data_quality():
        """Return quality metrics."""
        return {
            'total_records': 1000,
            'null_count': 50,
            'duplicate_count': 10,
            'quality_score': 0.94
        }
    
    @task
    def process_high_quality(data):
        """Handle high-quality data."""
        score = data['quality_score']
        print(f'High quality data (score: {score})')
        return {'path': 'standard_pipeline', 'data': data}
    
    @task
    def clean_and_process(data):
        """Handle low-quality data."""
        score = data['quality_score']
        print(f'Low quality data (score: {score}), cleaning...')
        cleaned_data = data.copy()
        cleaned_data['quality_score'] = 0.99
        cleaned_data['cleaned'] = True
        return {'path': 'cleaning_pipeline', 'data': cleaned_data}
    
    @task
    def final_load(result):
        """Load processed data."""
        path = result['path']
        print(f'Loading via {path}')
        return 'loaded'
    
    quality = check_data_quality()
    
    # Conditional branching based on data
    if quality['quality_score'] >= 0.95:
        result = process_high_quality(quality)
    else:
        result = clean_and_process(quality)
    
    final_load(result)

dag_instance = conditional_dag()

## 7. Data Validation and Schema Enforcement

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime

@dag(
    dag_id='data_validation',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def validation_dag():
    """Validate data structure and types."""
    
    @task
    def extract_raw():
        """Extract data (may have issues)."""
        return {
            'records': [
                {'id': 1, 'name': 'Alice', 'age': 30},
                {'id': 2, 'name': 'Bob', 'age': None},
                {'id': 3, 'name': '', 'age': 25}
            ]
        }
    
    @task
    def validate_schema(data):
        """Enforce schema requirements."""
        records = data['records']
        errors = []
        valid_records = []
        
        for i, record in enumerate(records):
            record_errors = []
            
            # Check required fields
            if not record.get('id'):
                record_errors.append('missing id')
            if not record.get('name'):
                record_errors.append('missing name')
            if record.get('age') is None:
                record_errors.append('missing age')
            
            if record_errors:
                errors.append({'record_index': i, 'errors': record_errors})
            else:
                valid_records.append(record)
        
        validation_result = {
            'total': len(records),
            'valid': len(valid_records),
            'invalid': len(errors),
            'errors': errors,
            'valid_records': valid_records
        }
        
        print(f'Validation: {validation_result["valid"]}/{validation_result["total"]} valid')
        return validation_result
    
    @task
    def process_valid(validation):
        """Process only valid records."""
        valid = validation['valid_records']
        print(f'Processing {len(valid)} valid records')
        
        # Add processing metadata
        for record in valid:
            record['processed'] = True
            record['processed_at'] = '2026-09-13'
        
        return valid
    
    raw = extract_raw()
    validated = validate_schema(raw)
    process_valid(validated)

dag_instance = validation_dag()

## 8. Aggregation Patterns

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime

@dag(
    dag_id='aggregation_pattern',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def aggregation_dag():
    """Aggregate results from parallel tasks."""
    
    @task
    def process_region_east():
        return {'region': 'east', 'sales': 50000, 'orders': 200}
    
    @task
    def process_region_west():
        return {'region': 'west', 'sales': 45000, 'orders': 180}
    
    @task
    def process_region_north():
        return {'region': 'north', 'sales': 38000, 'orders': 150}
    
    @task
    def process_region_south():
        return {'region': 'south', 'sales': 42000, 'orders': 170}
    
    @task
    def aggregate_results(east, west, north, south):
        """Combine all regional results."""
        regions = [east, west, north, south]
        
        total_sales = sum(r['sales'] for r in regions)
        total_orders = sum(r['orders'] for r in regions)
        avg_order_value = total_sales / total_orders if total_orders > 0 else 0
        
        summary = {
            'regions': regions,
            'total_sales': total_sales,
            'total_orders': total_orders,
            'avg_order_value': round(avg_order_value, 2),
            'top_region': max(regions, key=lambda x: x['sales'])['region']
        }
        
        print(f'Total Sales: ${total_sales:,}')
        print(f'Total Orders: {total_orders:,}')
        print(f'Avg Order Value: ${avg_order_value:.2f}')
        print(f'Top Region: {summary["top_region"]}')
        
        return summary
    
    east = process_region_east()
    west = process_region_west()
    north = process_region_north()
    south = process_region_south()
    
    aggregate_results(east, west, north, south)

dag_instance = aggregation_dag()

## 9. Serialization Considerations

In [ ]:
from airflow.decorators import dag, task
from datetime import datetime
import json

@dag(
    dag_id='serialization_examples',
    start_date=datetime(2026, 9, 1),
    schedule_interval='@daily',
    catchup=False
)
def serialization_dag():
    """Understand XCom serialization limits."""
    
    @task
    def supported_types():
        """These types serialize automatically."""
        return {
            'string': 'text',
            'integer': 42,
            'float': 3.14,
            'boolean': True,
            'list': [1, 2, 3],
            'dict': {'key': 'value'},
            'none': None,
            'datetime_str': '2026-09-13T10:00:00'
        }
    
    @task
    def handle_large_data():
        """For large datasets, return reference instead."""
        # DON'T: Return massive DataFrame directly
        # DO: Save to storage and return path
        
        file_path = '/data/output/large_dataset.parquet'
        # In real scenario: df.to_parquet(file_path)
        
        return {
            'file_path': file_path,
            'row_count': 1000000,
            'format': 'parquet'
        }
    
    @task
    def process_reference(data_ref):
        """Load data from storage reference."""
        path = data_ref['file_path']
        print(f'Loading data from: {path}')
        print(f'Expected rows: {data_ref["row_count"]:,}')
        # In real scenario: df = pd.read_parquet(path)
        return 'loaded'
    
    supported = supported_types()
    large_ref = handle_large_data()
    process_reference(large_ref)

dag_instance = serialization_dag()

## Summary

### Automatic XCom Features

| Feature | Traditional | TaskFlow |
|---------|------------|----------|
| Push data | `xcom_push()` | `return value` |
| Pull data | `xcom_pull()` | Function argument |
| Dependencies | `set_upstream()` | Function call |
| Multiple outputs | Multiple `xcom_push()` | `multiple_outputs=True` |

### Supported Data Types

✓ Strings, integers, floats, booleans  
✓ Lists and dictionaries  
✓ Nested structures  
✓ Pandas DataFrames (with serialization)  
✓ JSON-serializable objects  

### Best Practices

✓ Keep XCom payloads small (<1MB recommended)  
✓ Use file references for large datasets  
✓ Validate data structure before processing  
✓ Handle None values gracefully  
✓ Use type hints for clarity  
✓ Document expected data formats